# Telco Churn Q&A Notebook

Use this notebook to inspect the project and ask natural-language questions about the live MySQL database. The notebook reuses the same configuration and agent code as the scripts in this repository.

## What this project does

- `db_config.py` builds the MySQL SQLAlchemy URL from environment variables and loads `.env` or `.env.example`.
- `database_test.py` verifies that Python can reach the MySQL database.
- `ai_agent.py` creates a LangChain SQL agent backed by Google Gemini and answers questions from the live database.
- `README.md` documents the setup and the existing command-line workflow.

This notebook gives you a more interactive entry point for analysis and question answering.

In [ ]:
from sqlalchemy import create_engine, inspect
from sqlalchemy.engine import make_url
from sqlalchemy.exc import SQLAlchemyError

from ai_agent import ask_question
from db_config import get_database_url

database_url = get_database_url()
engine = create_engine(database_url)
inspector = inspect(engine)

print("Connected to:", make_url(database_url).render_as_string(hide_password=True))

In [ ]:
try:
    tables = inspector.get_table_names()
    print("Tables in the database:")
    for table_name in tables:
        print("-", table_name)

    print()
    for table_name in tables:
        print(f"Schema for {table_name}:")
        for column in inspector.get_columns(table_name):
            print(f"  - {column['name']} ({column['type']})")
        print()
except SQLAlchemyError as exc:
    print("Could not inspect the database schema.")
    print(exc)

In [ ]:
def ask(question: str) -> str:
    """Ask the project agent a natural-language question."""
    question = question.strip()
    if not question:
        raise ValueError("Question cannot be empty.")

    try:
        return ask_question(question)
    except SQLAlchemyError as exc:
        return (
            "MySQL connection failed. Check MYSQL_HOST, MYSQL_PORT, MYSQL_USER, "
            "MYSQL_PASSWORD, and MYSQL_DATABASE.\n"
            f"Details: {exc}"
        )

question = input("Ask a question about the Telco churn project: ")
print(ask(question))

## Example questions

- How many customers are in the database?
- What is the churn rate?
- Which customer groups churn the most?
- What patterns stand out in monthly charges versus churn?
- Show me a summary of the main tables and their purpose.